In [2]:
import pandas as pd
import numpy as np

# 1. CARICAMENTO E PREPARAZIONE DATI
df = pd.read_csv("/content/Historical Product Demand .csv")

# Pulizia Order_Demand: sostituzione parentesi con segno meno e conversione a numerico[cite: 1, 2]
df["Order_Demand"] = (
    df["Order_Demand"]
    .astype(str)
    .str.strip()
    .str.replace("(", "-", regex=False)
    .str.replace(")", "", regex=False)
)
df["Order_Demand"] = pd.to_numeric(df["Order_Demand"])

# Pulizia Date: conversione e filtro per il periodo affidabile 2012-2016[cite: 2]
df["Date"] = pd.to_datetime(df["Date"], format="%Y/%m/%d")
df = df.dropna(subset=["Date"])
df_analisi = df[(df["Date"] >= "2012-01-01") & (df["Date"] <= "2016-12-31")].copy()

# 2. CALCOLO DOMANDA MEDIA GIORNALIERA REALE
# Sommiamo tutta la domanda nel periodo per singolo prodotto
domanda_totale = df_analisi.groupby("Product_Code")["Order_Demand"].sum().reset_index()

# Calcoliamo i giorni totali del periodo (dal 2012-01-01 al 2016-12-31 sono 1827 giorni)[cite: 2]
giorni_totali = (pd.to_datetime("2016-12-31") - pd.to_datetime("2012-01-01")).days + 1

# Calcoliamo la media giornaliera
domanda_totale["Avg_Daily_Demand"] = (domanda_totale["Order_Demand"] / giorni_totali).round(1)

# Teniamo solo i prodotti con una domanda media positiva per le nostre regole di riordino
df_business = domanda_totale[domanda_totale["Avg_Daily_Demand"] > 0].copy()

# 3. SIMULAZIONE DATI DI MAGAZZINO
np.random.seed(42)
n_prodotti = len(df_business)
# Simuliamo uno stock coerente con i volumi di vendita
df_business['Current_Stock'] = (df_business['Avg_Daily_Demand'] * np.random.uniform(10, 60, size=n_prodotti)).astype(int)
df_business['Lead_Time_Days'] = np.random.choice([7, 14, 21, 30], size=n_prodotti)
df_business['Min_Order_Qty'] = np.random.choice([50, 100, 500, 1000], size=n_prodotti)

# 4. APPLICAZIONE REGOLE DI BUSINESS
# A. Scorta di Sicurezza (Copre il 50% della domanda durante il Lead Time)
df_business['Safety_Stock'] = (df_business['Avg_Daily_Demand'] * df_business['Lead_Time_Days'] * 0.5).astype(int)

# B. Punto di Riordino (Domanda nel Lead Time + Scorta di Sicurezza)
df_business['Reorder_Point'] = (df_business['Avg_Daily_Demand'] * df_business['Lead_Time_Days']).astype(int) + df_business['Safety_Stock']

# C. Monitoraggio Scorte e Autonomia
df_business['Stock_Operativo'] = df_business['Current_Stock'] - df_business['Safety_Stock']
df_business['Days_to_Safety_Stock'] = (df_business['Stock_Operativo'] / df_business['Avg_Daily_Demand']).round(1)
df_business['Total_Days_to_Zero'] = (df_business['Current_Stock'] / df_business['Avg_Daily_Demand']).round(1)

# D. Calcolo Quantità d'Ordine
df_business['Needs_Reorder'] = df_business['Current_Stock'] <= df_business['Reorder_Point']
quantita_teorica = df_business['Reorder_Point'] - df_business['Current_Stock']

df_business['Order_Quantity'] = np.where(
    df_business['Needs_Reorder'],
    np.maximum(quantita_teorica, df_business['Min_Order_Qty']),
    0
).astype(int)

# E. Semafori di Urgenza
def calcola_urgenza(row):
    if not row['Needs_Reorder']:
        return "🟢 OK"
    if row['Total_Days_to_Zero'] <= row['Lead_Time_Days']:
        return "🔴 CRITICO: Stock-Out"
    elif row['Days_to_Safety_Stock'] <= 0:
        return "🟠 ALTO: Intaccata Sicurezza"
    else:
        return "🟡 NORMALE: Da Ordinare"

df_business['Urgency'] = df_business.apply(calcola_urgenza, axis=1)

# 5. OUTPUT CRUSCOTTO FINALE
colonne_finali = [
    'Product_Code', 'Current_Stock', 'Safety_Stock', 'Reorder_Point',
    'Lead_Time_Days', 'Total_Days_to_Zero', 'Urgency', 'Order_Quantity'
]
df_final = df_business[colonne_finali].sort_values('Total_Days_to_Zero')

print("--- TOP 10 PRODOTTI CRITICI DA ORDINARE ---")
display(df_final.head(10))

--- TOP 10 PRODOTTI CRITICI DA ORDINARE ---


,Product_Code,Current_Stock,Safety_Stock,Reorder_Point,Lead_Time_Days,Total_Days_to_Zero,Urgency,Order_Quantity
1087,Product_1098,8,2,7,7,10.0,🟢 OK,0
482,Product_0483,2,0,1,7,10.0,🟢 OK,0
499,Product_0500,1,0,0,7,10.0,🟢 OK,0
2048,Product_2061,3,4,13,30,10.0,🔴 CRITICO: Stock-Out,500
1928,Product_1940,3,3,9,21,10.0,🔴 CRITICO: Stock-Out,100
1935,Product_1947,1,1,4,30,10.0,🔴 CRITICO: Stock-Out,500
948,Product_0959,1,1,3,21,10.0,🔴 CRITICO: Stock-Out,50
933,Product_0944,1,0,1,14,10.0,🔴 CRITICO: Stock-Out,100
1763,Product_1774,15,10,31,14,10.0,🔴 CRITICO: Stock-Out,100
806,Product_0807,2,3,9,30,10.0,🔴 CRITICO: Stock-Out,50
